In [ ]:
import pandas as pd
import numpy as np
import pyreadr
import optuna
import yaml
from typing import Optional
from optuna import trial
from IPython.display import clear_output

# Modelling
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score,make_scorer, root_mean_squared_error, ConfusionMatrixDisplay
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import GridSearchCV, KFold
from scipy.stats import randint
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import sklearn
from copy import deepcopy
from xgboost import XGBRegressor
from functools import partial



from sklearn.model_selection import cross_val_score, train_test_split,cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_squared_error

# Models
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
import os


In [ ]:
trial.BaseTrial

In [ ]:

train:pd.DataFrame = pyreadr.read_r('datasets/train.rds')[None]
rs_seed=1


In [ ]:
class YMLstudy():
    
    def __init__(self, model:str):

        os.makedirs("optuna-study", exist_ok=True)

        self.model = model
        self.file_name = f'{model}-opt_parm.yml'
        self.location = f'optuna-study/{self.file_name}'
        
        if os.path.isfile(self.location):
            self._get_vals()
    
    def read(self, what:Optional[str] = None):

            if what in ['all', None]:
                return self.best_all 
            else:
                return self.best_all[what]


    def write_study(self, study: optuna.Study):
        """Save only best_trials to YAML file."""
        with open(self.location, "w") as writer:
            pareto_data = []
            for t in study.best_trials:
                pareto_data.append({
                    "trial_number": t.number,
                    "values": [round(v, 4) for v in (
                        t.values if isinstance(t.values, (list, tuple)) else [t.value]
                    )],
                    "params": t.params,
                })

            yaml.safe_dump(
                {"best_trials": pareto_data},
                writer,
                default_flow_style=False,
                sort_keys=False,
            )

        self._get_vals()


    def _get_vals(self):
        """Load the first best trial from YAML (for quick access)."""
        with open(self.location, "r") as loader:
            out = yaml.safe_load(loader)
            self.best_all = out

            # Use only the first best trial
            first_trial = out["best_trials"][0] if out.get("best_trials") else None
            if first_trial:
                self.best_idx = first_trial["trial_number"]
                self.best_parameters = first_trial["params"]
                self.results = first_trial["values"]


def Search(model:str,
           df:pd.DataFrame,
           scores = {
                "RMSE": make_scorer(root_mean_squared_error, greater_is_better=False),
                # "R2": make_scorer(r2_score, greater_is_better=True)
            },
            directions:list[str] = ['minimize',
                                    # 'maximize'
                                    ],
            scale:bool=False,
            trials:int=1000,
            rs_seed:int=1) -> optuna.study:
    """
    Universal GridSearchCV function for regression models with preprocessing.

    Parameters
    ----------
    model : sklearn estimator
        The model to tune (e.g. RandomForestRegressor()).
    par_grid : dict
        Parameter grid for GridSearchCV (keys must include 'Model__').
    df : pd.DataFrame
        Input DataFrame containing predictors + 'score' column as target.
    scale : bool, default=True
        Whether to scale numerical features.
    rs_seed : int, default=1
        Random state for reproducibility.

    Returns
    -------
    search : GridSearchCV object (fitted)
    """
    match model:
        case 'RF':
            ml_model = RandomForestRegressor(bootstrap=True,
                            criterion='squared_error',
                            random_state=rs_seed)
            parameters = ParamRF
        case 'DTR':
            ml_model = DecisionTreeRegressor(splitter="best",
                            max_leaf_nodes=None,
                            random_state=rs_seed)
            parameters = ParamDTR

        case 'XGB':
            ml_model = XGBRegressor(random_state=rs_seed)
            parameters = ParamXGB

    X = deepcopy(df.drop(columns=["score"]))
    y = deepcopy(df["score"])

    cats = X.select_dtypes(include=["object", "category"]).columns.tolist()
    nums = X.select_dtypes(exclude=["object", "category"]).columns.tolist()

    preprocessor = ColumnTransformer([
        ("categorical", OneHotEncoder(sparse_output=False, handle_unknown="ignore"), cats),
        ("numeric", StandardScaler() if scale else "passthrough", nums)
    ])

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("Model", ml_model)
    ])
    ### I need to fix X
    yml = YMLstudy(model)
    study = _set_study(model, directions)

    objF = partial(OptunaObj,
                   data = (X,y),
                   pipe = pipeline,
                   scores=scores,
                   parameters=parameters)
    
    study.optimize(objF, n_trials=trials)

    clear_output()

    yml.write_study(study)

    return study

def _set_study(model, directions):
    if len(directions) > 1 and isinstance(directions,list):
        study = optuna.create_study(
            study_name=f"{model}_score",
            storage="sqlite:///optuna-study/optuna-study.db",
            directions=directions, # multi-objective
            load_if_exists=True,
        )
    else:
        study = optuna.create_study(
            study_name=f"{model}_score",
            storage="sqlite:///optuna-study/optuna-study.db",
            direction=directions[0] if isinstance(directions,list) else directions,      
            load_if_exists=True,
        )
    return study


def OptunaObj(
        trial,
        data:tuple,
        pipe:Pipeline,
        scores:dict,
        parameters):
    
    x,y=data
    print(scores)
    pipe.set_params(**parameters(trial))

    res = cross_validate(
        estimator=pipe,
        X=x,
        y=y,
        cv=KFold(n_splits=6, shuffle=True, random_state=rs_seed),
        scoring=scores,
        n_jobs=-1,
        verbose=0
    )

    rmse = -np.mean(res["test_RMSE"]) #becase cross_validate
    # r2 = np.mean(res["test_R2"])

    return rmse#,r2

def ParamXGB(trial):
    params = {
        "Model__n_estimators": trial.suggest_int("Model__n_estimators", 400, 1200, step=20),
        "Model__learning_rate": trial.suggest_float("Model__learning_rate", 0.01, 0.2, step=0.005),
        "Model__max_depth": trial.suggest_int("Model__max_depth", 2, 8),
        "Model__subsample": trial.suggest_float("Model__subsample", 0.5, 1.0),
        "Model__colsample_bytree": trial.suggest_float("Model__colsample_bytree", 0.5, 1.0),
        "Model__reg_lambda": trial.suggest_float("Model__reg_lambda", 1e-3, 10, log=True),   # log-scaled
        "Model__reg_alpha": trial.suggest_float("Model__reg_alpha", 1e-3, 10, log=True),     # log-scaled
        "Model__min_child_weight": trial.suggest_int("Model__min_child_weight", 2, 8),
        "Model__gamma": trial.suggest_float("Model__gamma", 0.0, 5.0),
        "Model__grow_policy": trial.suggest_categorical("Model__grow_policy", ["lossguide", "depthwise"]),
    }
    return params



def ParamRF(trial):
    params = {
        "Model__n_estimators": trial.suggest_int("Model__n_estimators", 200, 1200, step=20),
        "Model__max_depth": trial.suggest_categorical(
            "Model__max_depth", [None] + list(range(3, 9))
        ),
        "Model__min_samples_split": trial.suggest_int("Model__min_samples_split", 2, 20),
        "Model__min_samples_leaf": trial.suggest_int("Model__min_samples_leaf", 8, 15),
        "Model__max_features": trial.suggest_categorical("Model__max_features", ["sqrt", 0.3, 1.0]),
        "Model__max_leaf_nodes": trial.suggest_categorical(
            "Model__max_leaf_nodes", [None] + list(range(40, 201, 5))
        ),
        "Model__min_impurity_decrease": trial.suggest_float(
            "Model__min_impurity_decrease", 0.0, 1e-8
        ),
        "Model__ccp_alpha": trial.suggest_float("Model__ccp_alpha", 0.0, 0.02),
    }

    return params

def ParamDTR(trial):
    params = {
        "Model__criterion": trial.suggest_categorical(
            "Model__criterion", ["squared_error", "friedman_mse"]
        ),
        "Model__max_depth": trial.suggest_int("Model__max_depth", 6, 25),
        "Model__min_samples_split": trial.suggest_int("Model__min_samples_split", 5, 25),
        "Model__min_samples_leaf": trial.suggest_int("Model__min_samples_leaf", 5, 18),
        "Model__max_features": trial.suggest_categorical(
            "Model__max_features", ["sqrt", 0.3, 0.5, 0.7, 1.0]
        ),
        "Model__ccp_alpha": trial.suggest_float("Model__ccp_alpha", 0.0, 0.05),
        "Model__min_impurity_decrease": trial.suggest_float("Model__min_impurity_decrease", 0.0, 0.01),
    }
    return params


In [31]:
xgb=Search('DTR',train,trials=2000)

In [32]:
xgb=Search('XGB',train,trials=2000)

In [34]:
rf=Search('RF',train,trials=2000)

[I 2025-10-16 23:33:53,463] Using an existing study with name 'RF_score' instead of creating a new one.


{'RMSE': make_scorer(root_mean_squared_error, greater_is_better=False, response_method='predict')}


[I 2025-10-16 23:33:55,487] Trial 246 finished with value: 0.847269403130359 and parameters: {'Model__n_estimators': 1160, 'Model__max_depth': None, 'Model__min_samples_split': 14, 'Model__min_samples_leaf': 8, 'Model__max_features': 1.0, 'Model__max_leaf_nodes': 75, 'Model__min_impurity_decrease': 3.455009538220668e-09, 'Model__ccp_alpha': 0.0192889992544264}. Best is trial 219 with value: 0.8471208339718479.


{'RMSE': make_scorer(root_mean_squared_error, greater_is_better=False, response_method='predict')}


[I 2025-10-16 23:33:57,310] Trial 247 finished with value: 0.8473262984991742 and parameters: {'Model__n_estimators': 1180, 'Model__max_depth': None, 'Model__min_samples_split': 15, 'Model__min_samples_leaf': 8, 'Model__max_features': 1.0, 'Model__max_leaf_nodes': 85, 'Model__min_impurity_decrease': 4.039064161352245e-09, 'Model__ccp_alpha': 0.018692857173723633}. Best is trial 219 with value: 0.8471208339718479.


...

{'RMSE': make_scorer(root_mean_squared_error, greater_is_better=False, response_method='predict')}


[I 2025-10-16 23:40:38,886] Trial 652 finished with value: 0.8473864560428531 and parameters: {'Model__n_estimators': 1020, 'Model__max_depth': 8, 'Model__min_samples_split': 11, 'Model__min_samples_leaf': 8, 'Model__max_features': 1.0, 'Model__max_leaf_nodes': 125, 'Model__min_impurity_decrease': 8.60071927125258e-09, 'Model__ccp_alpha': 0.019963198631204656}. Best is trial 622 with value: 0.8470913235969794.


{'RMSE': make_scorer(root_mean_squared_error, greater_is_better=False, response_method='predict')}


[W 2025-10-16 23:40:40,077] Trial 653 failed with parameters: {'Model__n_estimators': 1040, 'Model__max_depth': 8, 'Model__min_samples_split': 11, 'Model__min_samples_leaf': 8, 'Model__max_features': 1.0, 'Model__max_leaf_nodes': 125, 'Model__min_impurity_decrease': 8.442849725662706e-09, 'Model__ccp_alpha': 0.019937157567566383} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/home/leonardo/Public/projects/dw-3/venv-dw3/lib/python3.11/site-packages/optuna/study/_optimize.py", line 201, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_43833/1296043237.py", line 169, in OptunaObj
    res = cross_validate(
          ^^^^^^^^^^^^^^^
  File "/home/leonardo/Public/projects/dw-3/venv-dw3/lib/python3.11/site-packages/sklearn/utils/_param_validation.py", line 218, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/home/leonardo/Public/projects/dw-3/venv-dw3

KeyboardInterrupt: 